# Notebook 01 — Tox21 Multi-task Benchmark: DeepTox · Tox21NN · D-MPNN
**Author: Himanshu Goel** | [Website](https://hgoelgithub.github.io)

---

## Background

### The Tox21 Initiative
The **Toxicology in the 21st Century (Tox21)** programme is a federal collaboration between the NIH, EPA, and FDA launched in 2008 to modernise toxicology testing. Its core goal is to replace slow, expensive animal bioassays with high-throughput *in vitro* assays that predict human toxicity from chemical structure alone — shifting the field from *in vivo* observation to *in silico* mechanism.

The **2014 Tox21 Data Challenge** released ~12 000 compounds measured across 12 nuclear-receptor and stress-response endpoints, inviting teams worldwide to build predictive models. Over 8 000 teams participated. The winner was **DeepTox** (Mayr et al. 2016) — one of the landmark early demonstrations that deep learning outperforms classical QSAR at a large public benchmark.

### The 12 Endpoints
| Panel | Endpoint | Assay Biology | Clinical Relevance |
|-------|----------|---------------|--------------------|
| Nuclear Receptor | **NR-AR** | Androgen receptor agonism | Endocrine disruption, prostate cancer |
| Nuclear Receptor | **NR-AR-LBD** | AR ligand-binding domain | Endocrine disruption |
| Nuclear Receptor | **NR-AhR** | Aryl hydrocarbon receptor | Dioxin-like toxicity, CYP1A1/1B1 induction |
| Nuclear Receptor | **NR-Aromatase** | CYP19A1 inhibition | Oestrogen biosynthesis disruption |
| Nuclear Receptor | **NR-ER** | Oestrogen receptor α agonism | Endocrine disruption, breast cancer |
| Nuclear Receptor | **NR-ER-LBD** | ERα ligand-binding domain | Endocrine disruption |
| Nuclear Receptor | **NR-PPAR-gamma** | PPARγ agonism | Lipid metabolism, adipogenesis |
| Stress Response | **SR-ARE** | Nrf2/antioxidant response element | Oxidative stress, electrophile reactivity |
| Stress Response | **SR-ATAD5** | ATAD5 genotoxicity proxy | DNA replication stress |
| Stress Response | **SR-HSE** | Heat-shock element (HSP70) | Protein stress, chaperone induction |
| Stress Response | **SR-MMP** | Mitochondrial membrane potential | Mitochondrial toxicity, uncouplers |
| Stress Response | **SR-p53** | p53 pathway activation | DNA damage, genotoxicity |

### The Class Imbalance & Missing Label Problem
Active rates range from **3% (NR-AR-LBD)** to **17% (SR-MMP)**. Compounding this, **30–50% of labels are missing per endpoint**: not every compound was assayed on every endpoint. The weight matrix `w[i,j]` encodes this: `w=0` means “not tested” — those (compound, task) pairs must be **masked from both loss and AUC**, never imputed as negatives.

### Three Methods Compared in This Notebook

| Method | Input | Key Idea | Reference |
|--------|-------|----------|-----------|
| **Tox21NN** | ECFP4 | Independent per-task shallow MLP baseline | Unterthiner et al. 2014 |
| **DeepTox** | ECFP4 | Shared deep trunk + per-task heads; multi-task | Mayr et al. 2016 |
| **D-MPNN** | Molecular graph | Directed message passing on atoms/bonds | Yang et al. 2019 (chemprop) |

**References**
- Mayr A et al. DeepTox. *Front Environ Sci* 2016; doi:10.3389/fenvs.2016.00080
- Unterthiner T et al. Toxicity Prediction using Deep Learning. NIPS Workshop 2014
- Yang K et al. Analyzing Learned Molecular Representations. *J Chem Inf Model* 2019; doi:10.1021/acs.jcim.9b00237
- Wu Z et al. MoleculeNet. *Chem Sci* 2018; doi:10.1039/C7SC02664A

In [14]:
!pip install deepchem rdkit scikit-learn pandas numpy matplotlib seaborn torch -q

In [ ]:
# =============================================================================
# CELL 1 — Environment setup, data loading
# =============================================================================

# -- SSL fix ------------------------------------------------------------------
# macOS Python.org installer does not ship with root CA certificates.
# Without this line, urllib raises SSLCertVerificationError when DeepChem tries
# to download the Tox21 dataset over HTTPS from MoleculeNet servers.
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

# -- Suppress RDKit deprecation warning ---------------------------------------
# DeepChem's ECFP featurizer internally calls GetMorganFingerprintAsBitVect,
# which RDKit deprecated in favour of MorganGenerator. Without silencing this,
# every molecule in the dataset prints a [HH:MM:SS] DEPRECATION WARNING,
# producing thousands of noisy log lines that obscure real errors.
from rdkit import RDLogger, Chem
from rdkit.Chem import rdchem
RDLogger.DisableLog('rdApp.warning')

# -- Standard imports ---------------------------------------------------------
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score
import deepchem as dc

warnings.filterwarnings('ignore')

# -- Reproducibility ----------------------------------------------------------
# Fix all random seeds so results are reproducible across runs.
# PyTorch uses separate CPU and CUDA generators, so both must be seeded.
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Use GPU if available; all tensors and models are sent to DEVICE throughout.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE} | PyTorch {torch.__version__} | DeepChem {dc.__version__}')

# -- Data loading -------------------------------------------------------------
# featurizer='ECFP': computes 1024-bit Morgan/ECFP4 fingerprints (radius=2).
#   Each bit indicates the presence of a specific circular substructure.
#   This is the standard baseline fingerprint in cheminformatics.
#
# splitter='scaffold': uses Bemis-Murcko scaffold split instead of random split.
#   Molecules with the same core ring system are kept in the same fold.
#   This prevents the model from simply memorising a scaffold and reporting
#   inflated AUC — random split inflates performance by 5-15% vs scaffold split.
print('Loading Tox21 (ECFP4, scaffold split)...')
tox21_tasks, datasets, transformers = dc.molnet.load_tox21(
    featurizer='ECFP', splitter='scaffold',
)
train_ds, valid_ds, test_ds = datasets

# Save original SMILES strings from the dataset IDs field.
# DeepChem's ECFP featurizer converts SMILES → bit-vector and discards the
# molecule object. We cache the SMILES here because Method 3 (D-MPNN) needs
# them later to build atom/bond graphs — once the featurizer runs, SMILES are
# only available via ds.ids.
smiles_train = list(train_ds.ids)
smiles_valid = list(valid_ds.ids)
smiles_test  = list(test_ds.ids)

print(f'Tasks ({len(tox21_tasks)}): {tox21_tasks}')
print(f'Train: {len(train_ds):,} | Valid: {len(valid_ds):,} | Test: {len(test_ds):,}')
print(f'ECFP4 feature dim: {train_ds.X.shape[1]}')

## Exploratory Data Analysis — Class Balance & Missing Labels

Tox21 labels are massively sparse. The weight matrix `w[i,j] = 0` marks untested
(compound, endpoint) pairs — these must be masked from loss and AUC in all three methods.

In [ ]:
# =============================================================================
# CELL 2 — Exploratory Data Analysis: class balance and missing labels
# =============================================================================

y_tr, w_tr = train_ds.y, train_ds.w   # y: labels (0/1), w: weight matrix (0/1)

stats = []
for i, task in enumerate(tox21_tasks):
    # w[compound, task] == 0 means the compound was NEVER TESTED on this endpoint.
    # It does NOT mean inactive. Treating it as 0 (negative) would artificially
    # double the negative class and destroy calibration for rare endpoints.
    mask  = w_tr[:, i] > 0            # select only measured (tested) compounds
    vals  = y_tr[:, i][mask]
    n_pos = int(vals.sum())
    n_neg = int((vals == 0).sum())
    n_mis = int((w_tr[:, i] == 0).sum())   # how many were untested
    stats.append({
        'Endpoint': task,
        'N_active':  n_pos,
        'N_inactive': n_neg,
        'N_missing':  n_mis,
        'Total':      n_pos + n_neg,
        # Active% computed only over measured compounds — not total — to give
        # the true assay hit rate rather than a diluted apparent hit rate.
        'Active%': round(n_pos / (n_pos + n_neg) * 100, 1) if (n_pos + n_neg) else 0
    })

df_eda = pd.DataFrame(stats).sort_values('Active%')
print(df_eda[['Endpoint','N_active','N_inactive','N_missing','Active%']].to_string(index=False))

# -- Visualisation ------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Left plot: active rate per endpoint.
# Colour-code by severity of imbalance — red (<8%) needs most care in training.
pal = ['#e74c3c' if r < 8 else '#f39c12' if r < 15 else '#27ae60'
       for r in df_eda['Active%']]
ax1.barh(df_eda['Endpoint'], df_eda['Active%'], color=pal)
# 15% guideline: below this, standard BCE loss is heavily dominated by negatives.
# Focal loss or class weighting becomes important below this threshold.
ax1.axvline(15, color='k', linestyle='--', lw=0.8, label='15% guideline')
ax1.set_xlabel('% Active (measured only)')
ax1.set_title('Class balance per Tox21 endpoint')
ax1.legend()

# Right plot: stacked bar showing measured vs. missing compounds per endpoint.
# The grey "Missing" bars visualise how sparse the label matrix is.
# A compound measured in 12/12 endpoints is rare — most are tested on 3-5 only.
xp = range(len(df_eda))
ax2.bar(xp, df_eda['Total'],     color='#2c3e50', alpha=0.8, label='Measured')
ax2.bar(xp, df_eda['N_missing'], bottom=df_eda['Total'],
        color='#bdc3c7', alpha=0.5, label='Missing')
ax2.set_xticks(list(xp))
ax2.set_xticklabels(df_eda['Endpoint'].tolist(), rotation=45, ha='right')
ax2.set_ylabel('Compounds')
ax2.set_title('Data availability per endpoint')
ax2.legend()

plt.tight_layout()
plt.savefig('tox21_eda.png', dpi=150)
plt.show()

In [ ]:
# =============================================================================
# CELL 3 — Enriched feature matrix: ECFP4 + MACCS keys + RDKit 2D descriptors
# =============================================================================
# Using three complementary fingerprint types consistently outperforms ECFP4
# alone by ~2-4% AUC on Tox21-class benchmarks:
#
#   ECFP4  (1024-dim): Morgan circular fingerprints, radius ≤ 2.
#     Encodes the local chemical environment of each atom out to 2 bonds.
#     Hashed into a 1024-bit vector — fast, widely used, good general coverage.
#
#   MACCS  ( 167-dim): Molecular ACCess System structural keys.
#     166 hand-crafted SMARTS rules: "does the molecule have a nitro group?",
#     "does it have an aromatic amine?", etc. Directly encodes known
#     toxicophore patterns that ECFP4 may collide/hash away.
#
#   RDKit  (  10-dim): Global physicochemical descriptors.
#     MW, LogP, TPSA, H-bond donors/acceptors, rotatable bonds, ring counts.
#     These govern membrane penetration, absorption, and receptor accessibility —
#     context that neither ECFP4 nor MACCS captures.

from rdkit.Chem import MACCSkeys, Descriptors
from rdkit.DataStructs import ConvertToNumpyArray   # RDKit → numpy without Python loop
from sklearn.preprocessing import StandardScaler

RDKIT_DESC_NAMES = [
    'MolWt', 'MolLogP', 'TPSA', 'NumHDonors', 'NumHAcceptors',
    'NumRotatableBonds', 'NumAromaticRings', 'RingCount', 'FractionCSP3',
    'NumHeteroatoms',
]

def _maccs(smi: str) -> np.ndarray:
    """Convert a SMILES to a 167-bit MACCS key vector."""
    mol = Chem.MolFromSmiles(smi)
    arr = np.zeros(167, dtype=np.float32)
    if mol:
        # ConvertToNumpyArray writes directly into the pre-allocated array —
        # faster than np.array(list(fp)) which unpacks bit-by-bit in Python.
        ConvertToNumpyArray(MACCSkeys.GenMACCSKeys(mol), arr)
    return arr   # returns all-zeros for unparseable SMILES (graceful degradation)

def _rdkit_descs(smi: str) -> np.ndarray:
    """Compute 10 global physicochemical descriptors for a SMILES string."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return np.zeros(len(RDKIT_DESC_NAMES), dtype=np.float32)
    row = []
    for name in RDKIT_DESC_NAMES:
        try:
            v = float(getattr(Descriptors, name)(mol))
            # Replace NaN/Inf (can occur for exotic molecules) with 0 — safer
            # than letting them propagate into the neural network weights.
            row.append(v if np.isfinite(v) else 0.0)
        except Exception:
            row.append(0.0)
    return np.array(row, dtype=np.float32)

def build_enriched(smiles_list: list, ecfp_matrix: np.ndarray) -> np.ndarray:
    """Concatenate ECFP4, MACCS, and RDKit 2D arrays row-wise."""
    maccs  = np.stack([_maccs(s)       for s in smiles_list])   # (N, 167)
    rdkit  = np.stack([_rdkit_descs(s) for s in smiles_list])   # (N, 10)
    # np.concatenate along axis=1 → (N, 1024+167+10) = (N, 1201)
    return np.concatenate([ecfp_matrix.astype(np.float32), maccs, rdkit], axis=1)

print('Building enriched features (ECFP4 + MACCS + RDKit 2D)...')
X_tr_enr = build_enriched(smiles_train, train_ds.X)
X_va_enr = build_enriched(smiles_valid, valid_ds.X)
X_te_enr = build_enriched(smiles_test,  test_ds.X)

# Standardise ONLY the continuous RDKit block (last 10 columns).
# ECFP4 and MACCS are binary {0,1} — standardising binary bits destroys their
# meaning and can make near-zero bits oscillate around 0 confusingly.
# Fit StandardScaler on TRAIN only to prevent test data leaking into the scaler
# statistics (mean and std must be unknown at test time).
scaler = StandardScaler()
X_tr_enr[:, -10:] = scaler.fit_transform(X_tr_enr[:, -10:]).astype(np.float32)
X_va_enr[:, -10:] = scaler.transform(X_va_enr[:, -10:]).astype(np.float32)
X_te_enr[:, -10:] = scaler.transform(X_te_enr[:, -10:]).astype(np.float32)

ENR_DIM = X_tr_enr.shape[1]   # 1201 — used by all fingerprint model definitions
print(f'Feature dim: {ENR_DIM}  ({train_ds.X.shape[1]} ECFP4 + 167 MACCS + 10 RDKit 2D)')

## Method 1 — Tox21NN (Per-task Fingerprint Baseline)

**Tox21NN** (Unterthiner et al. 2014) trains an independent shallow MLP for each
endpoint separately on fingerprint features. No cross-task signal — each endpoint is
solved in isolation. Fastest and simplest of the three methods.

**Improvement:** enriched input — ECFP4 (1024) + MACCS keys (167) + RDKit 2D (10) = **1201-dim**.
MACCS keys capture explicit functional-group patterns (nitro, amine, halogen…) that ECFP4
often hashes away; RDKit 2D adds global physicochemical context (lipophilicity, H-bond capacity).

Architecture per endpoint: `Enriched(1201) → BN→ReLU→Drop(512) → BN→ReLU→Drop(256) → 1`

In [ ]:
# =============================================================================
# CELL 4 — Method 1: Tox21NN — independent per-task MLPs
# =============================================================================

class Tox21NN(nn.Module):
    """Shallow MLP for a single binary classification endpoint.

    Architecture: Enriched(1201) → BN→ReLU→Drop(512) → BN→ReLU→Drop(256) → 1
    BatchNorm before ReLU stabilises training on sparse binary inputs where
    feature magnitudes can vary widely across the 1201 dimensions.
    Dropout prevents overfitting, especially on small per-endpoint datasets.
    """
    def __init__(self, in_features: int, dropout: float = 0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, 256),         nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 1),   # raw logit — no sigmoid here; applied at eval time
        )
    def forward(self, x): return self.net(x)


# Convert enriched numpy arrays to CUDA/CPU tensors once.
# These same tensors are reused in the DeepTox cell, avoiding redundant copies.
X_tr_fp = torch.FloatTensor(X_tr_enr).to(DEVICE)
X_va_fp = torch.FloatTensor(X_va_enr).to(DEVICE)
X_te_fp = torch.FloatTensor(X_te_enr).to(DEVICE)

tox21nn_aucs = {}   # accumulate per-task test AUC for final comparison table

# Train one completely independent model per endpoint.
# This is the "no multi-task" baseline — no cross-endpoint information transfer.
for task_idx, task_name in enumerate(tox21_tasks):

    # Select only compounds that were MEASURED on this endpoint (w > 0).
    # Compounds with w=0 were never tested — including them as negatives would
    # corrupt the label distribution and depress the AUC artificially.
    mask_tr = train_ds.w[:, task_idx] > 0
    if mask_tr.sum() < 20:
        continue   # skip endpoint if training data is too sparse to be useful

    Xt = X_tr_fp[mask_tr]
    # unsqueeze(1) reshapes labels from (N,) to (N,1) to match the model's
    # output shape (N,1), required by binary_cross_entropy_with_logits.
    yt = torch.FloatTensor(train_ds.y[mask_tr, task_idx]).unsqueeze(1).to(DEVICE)

    model_st  = Tox21NN(ENR_DIM).to(DEVICE)
    opt_st    = torch.optim.Adam(model_st.parameters(), lr=1e-3, weight_decay=1e-5)
    # weight_decay adds L2 regularisation to all weights — penalises large weights
    # to reduce overfitting, especially important for small per-task datasets.
    loader_st = DataLoader(TensorDataset(Xt, yt), batch_size=128, shuffle=True,
                           generator=torch.Generator().manual_seed(SEED))

    for epoch in range(15):
        model_st.train()
        for xb, yb in loader_st:
            opt_st.zero_grad()
            # binary_cross_entropy_with_logits = sigmoid + BCE in one numerically
            # stable step. Avoids float underflow that occurs when you apply
            # torch.sigmoid() first and then compute log(p) separately.
            F.binary_cross_entropy_with_logits(model_st(xb), yb).backward()
            # Gradient clipping prevents exploding gradients (norm > 1.0 gets
            # scaled down). Especially useful with BatchNorm + Dropout.
            torch.nn.utils.clip_grad_norm_(model_st.parameters(), 1.0)
            opt_st.step()

    # Evaluate on test set —————————————————————————————————————————————
    model_st.eval()
    with torch.no_grad():   # disable autograd to save memory during inference
        # sigmoid converts logits → probabilities in [0,1] for roc_auc_score
        te_probs = torch.sigmoid(model_st(X_te_fp)).cpu().numpy().flatten()

    mask_te = test_ds.w[:, task_idx] > 0
    lbl     = test_ds.y[mask_te, task_idx]
    # Guard: need at least 5 samples AND both classes present to compute AUC.
    # roc_auc_score raises ValueError if only one class is in y_true.
    if mask_te.sum() >= 5 and len(np.unique(lbl)) > 1:
        try: tox21nn_aucs[task_name] = roc_auc_score(lbl, te_probs[mask_te])
        except: pass
    print(f'  {task_name:20s}: {tox21nn_aucs.get(task_name, float("nan")):.4f}')

print(f'Tox21NN mean test AUC: {np.mean(list(tox21nn_aucs.values())):.4f}')

## Method 2 — DeepTox (Multi-task Deep MLP)

**DeepTox** (Mayr et al. 2016) won the 2014 Tox21 Challenge with a single deep network
where all 12 endpoints share a common representation trunk, then branch into per-task heads.

**Improvements in this notebook:**
| Change | Why |
|--------|-----|
| **Focal loss** (γ=2, α=0.75) | Down-weights easy negatives; focuses gradients on hard active compounds (Lin et al. 2017) |
| **3-seed ensemble** | Average predictions across seeds 42/0/1 — reduces variance, typically +1–2% AUC |
| **Enriched features** (1201-dim) | ECFP4 + MACCS + RDKit 2D (same as Tox21NN) |
| **Cosine LR schedule** | Smoother optimisation than step-decay; avoids abrupt restarts |

Architecture: `Enriched(1201) → [2048→1024→512→256] shared trunk → 12 × task head`

In [ ]:
# =============================================================================
# CELL 5 — Method 2: DeepTox — focal loss + multi-task MLP + 3-seed ensemble
# =============================================================================

# -- Focal Loss ---------------------------------------------------------------
# Standard BCE treats every (compound, task) pair equally regardless of
# confidence. With 3-17% active rates, the model sees ~20x more negatives than
# positives. Easy negatives dominate the gradient, causing the model to become
# very confident at predicting "inactive" while barely learning rare actives.
#
# Focal loss (Lin et al. 2017) adds two corrections:
#   alpha: static class-prior correction — upweights positives by alpha=0.75
#          (positives weighted 3x more than negatives on average)
#   (1-pt)^gamma: dynamic focusing — pt is the model's probability for the
#          CORRECT class. When pt → 1 (confident correct), (1-pt)^2 → 0,
#          killing that sample's gradient contribution. When pt → 0.5 (uncertain
#          or wrong), (1-pt)^2 ≈ 0.25, preserving full gradient. Result: the
#          model ignores easy examples and concentrates learning on hard ones.
def masked_focal(pred, target, weight, gamma: float = 2.0, alpha: float = 0.75):
    mask = weight > 0             # exclude untested (w=0) compound-endpoint pairs
    if not mask.any(): return (pred * 0.0).sum()
    p  = torch.sigmoid(pred[mask])
    ce = F.binary_cross_entropy_with_logits(pred[mask], target[mask], reduction='none')
    pt = torch.where(target[mask] == 1, p, 1 - p)   # prob of the true class
    at = torch.where(target[mask] == 1,
                     torch.full_like(pt, alpha),
                     torch.full_like(pt, 1.0 - alpha))   # per-sample class weight
    return (at * (1 - pt) ** gamma * ce).mean()


# -- DeepTox architecture -----------------------------------------------------
class DeepToxNet(nn.Module):
    """Multi-task deep MLP: one shared trunk + 12 per-task output heads.

    The shared trunk forces all 12 endpoints to use the same intermediate
    representation. This means gradients from SR-ARE (7,000+ positives) flow
    through the same weights as NR-AR (300 positives) — the rare endpoint
    borrows representational capacity from the data-rich endpoint.
    This is the core insight of multi-task learning for sparse bioassay data.
    """
    def __init__(self, in_features: int, n_tasks: int,
                 hidden: list = None, dropout: float = 0.35):
        super().__init__()
        if hidden is None:
            hidden = [2048, 1024, 512, 256]   # DeepTox paper architecture
        layers, dim = [], in_features
        for h in hidden:
            # Each trunk block: Linear → BatchNorm → ReLU → Dropout.
            # Deeper dropout (0.35) is used here vs Tox21NN (0.25) because the
            # trunk is much wider (2048 units) — more aggressive regularisation
            # is needed to prevent the large first layer from memorising the data.
            layers += [nn.Linear(dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            dim = h
        self.trunk = nn.Sequential(*layers)
        # One linear head per task — final 256-dim trunk representation → scalar logit.
        # Each head has its own gradient signal from its own endpoint's labels.
        self.heads = nn.ModuleList([nn.Linear(dim, 1) for _ in range(n_tasks)])

    def forward(self, x):
        h = self.trunk(x)
        # cat along dim=1 → output shape (batch, 12) — one logit per endpoint
        return torch.cat([head(h) for head in self.heads], dim=1)


y_tr_t = torch.FloatTensor(train_ds.y).to(DEVICE)
w_tr_t = torch.FloatTensor(train_ds.w).to(DEVICE)

# -- 3-seed ensemble training -------------------------------------------------
# Each seed produces a model that converged to a different local minimum, having
# seen the data in a different random order with different weight initialisations.
# Individual model errors are partially uncorrelated across seeds, so averaging
# their probability estimates cancels variance without any extra data.
ENSEMBLE_SEEDS = [42, 0, 1]
EPOCHS_DT      = 25
ensemble_te_probs, seed_val_aucs = [], []

for run_seed in ENSEMBLE_SEEDS:
    torch.manual_seed(run_seed)
    np.random.seed(run_seed)

    model_dt = DeepToxNet(ENR_DIM, len(tox21_tasks)).to(DEVICE)
    opt_dt   = torch.optim.Adam(model_dt.parameters(), lr=1e-3, weight_decay=1e-5)

    # CosineAnnealingLR decays lr from 1e-3 → ~0 following a cosine curve over
    # EPOCHS_DT steps. This provides large updates early (exploration phase) and
    # tiny refinements late (exploitation phase), avoiding the sharp loss spikes
    # that ReduceLROnPlateau can cause when it steps at the wrong moment.
    sched_dt = torch.optim.lr_scheduler.CosineAnnealingLR(opt_dt, T_max=EPOCHS_DT)

    # DataLoader shuffles training data each epoch (shuffle=True).
    # Passing the generator with the run_seed ensures each seed also has a
    # different data order — important for ensemble diversity.
    loader_dt = DataLoader(TensorDataset(X_tr_fp, y_tr_t, w_tr_t), batch_size=256,
                           shuffle=True, generator=torch.Generator().manual_seed(run_seed))
    best_auc_dt, best_state_dt = 0.0, None

    for epoch in range(EPOCHS_DT):
        model_dt.train()
        for xb, yb, wb in loader_dt:
            opt_dt.zero_grad()
            masked_focal(model_dt(xb), yb, wb).backward()
            torch.nn.utils.clip_grad_norm_(model_dt.parameters(), 1.0)
            opt_dt.step()
        sched_dt.step()   # advance cosine schedule by one epoch

        # Validation AUC checkpoint — save the state that achieves the best
        # mean AUC across all 12 tasks on the validation fold. This prevents
        # saving a state where one task AUC spiked while others regressed.
        model_dt.eval()
        with torch.no_grad():
            vp = torch.sigmoid(model_dt(X_va_fp)).cpu().numpy()
        aucs = []
        for i in range(len(tox21_tasks)):
            m = valid_ds.w[:, i] > 0
            lbl = valid_ds.y[:, i][m]
            if m.sum() >= 10 and len(np.unique(lbl)) > 1:
                try: aucs.append(roc_auc_score(lbl, vp[:, i][m]))
                except: pass
        mean_va = float(np.mean(aucs)) if aucs else 0.0
        if mean_va > best_auc_dt:
            best_auc_dt = mean_va
            # Deep-copy state_dict so later epochs don't overwrite this snapshot
            best_state_dt = {k: v.clone() for k, v in model_dt.state_dict().items()}

    # Restore the best checkpoint, then evaluate on test
    model_dt.load_state_dict(best_state_dt)
    model_dt.eval()
    with torch.no_grad():
        tp = torch.sigmoid(model_dt(X_te_fp)).cpu().numpy()
    ensemble_te_probs.append(tp)          # store this seed's test probabilities
    seed_val_aucs.append(best_auc_dt)
    print(f'  Seed {run_seed}: best val AUC = {best_auc_dt:.4f}')

# Average probability estimates (not logits) across the 3 seeds.
# Operating in probability space (after sigmoid) means each model's confidence
# is equally weighted — averaging logits would bias toward any single model's scale.
tp_ens = np.mean(ensemble_te_probs, axis=0)   # shape: (N_test, 12)

deeptox_aucs = {}
for i, task in enumerate(tox21_tasks):
    m = test_ds.w[:, i] > 0
    lbl = test_ds.y[:, i][m]
    if m.sum() >= 5 and len(np.unique(lbl)) > 1:
        try: deeptox_aucs[task] = roc_auc_score(lbl, tp_ens[:, i][m])
        except: pass
print(f'\nDeepTox (3-seed ensemble + focal loss) mean test AUC: {np.mean(list(deeptox_aucs.values())):.4f}')

## Method 3 — D-MPNN (Directed Message Passing Neural Network)

**D-MPNN** (Yang et al. 2019, *chemprop*) operates on the **molecular graph** rather
than a pre-hashed fingerprint. This is the key architectural distinction:

| Property | ECFP4 (DeepTox / Tox21NN) | D-MPNN |
|----------|---------------------------|--------|
| Input | Fixed 1024-bit vector | Atom + bond feature tensors |
| Feature radius | Hard-coded radius-2 | Learnable message depth |
| Bond information | Implicit (hashed) | Explicit bond-type features |
| Generalisation | Fixed structural vocabulary | Inductive over new substructures |

### How D-MPNN Works
1. **Initialise** each directed edge `(v→w)`: `h⁰_vw = ReLU(W_i · [x_v ‖ e_vw])` — atom + bond features
2. **Message pass** for *T* steps:
   - `mᵗ_vw = Σ_{u ∈ N(v) \ w} hᵗ⁻¹_{u→v}` — aggregate neighbours, **exclude reverse edge** (directed!)
   - `hᵗ_vw = ReLU(h⁰_vw + W_h · mᵗ_vw)` — residual update
3. **Atom readout**: `h_v = ReLU(W_o · [x_v ‖ Σ_w hᵀ_{v→w}])`
4. **Molecule embedding**: `h_mol = Σ_v h_v` (sum pooling over atoms)
5. **Task heads**: per-task FFN applied to `h_mol`

**Improvements in this notebook:** `depth=5` (vs. paper's 3), `hidden=300` (vs. 256),
focal loss, cosine LR schedule, 30 epochs — giving the graph model more capacity and
longer convergence time to exploit structural information beyond ECFP's radius-2 cutoff.

For production use: `pip install chemprop` for the fully optimised batched version.

In [ ]:
# =============================================================================
# CELL 6 — D-MPNN featurisation: molecules → atom/bond graph tensors
# =============================================================================
# Unlike ECFP4 which hashes substructures into a fixed bit-vector, D-MPNN
# operates directly on the molecular graph where atoms are nodes and bonds are
# directed edges. This allows the model to LEARN which structural patterns matter
# rather than being constrained by a fixed radius-2 hash.

# Atom types included in the one-hot encoding. "Other" (index 10) catches
# everything else (e.g. Si, Se, metals) — important for Tox21 which includes
# organometallics. More common atoms get their own dimension; rare ones share "Other".
COMMON_ATOMS = [1, 6, 7, 8, 9, 15, 16, 17, 35, 53]   # H C N O F P S Cl Br I

HYB = [rdchem.HybridizationType.SP, rdchem.HybridizationType.SP2,
       rdchem.HybridizationType.SP3, rdchem.HybridizationType.OTHER]

def onek(val, choices):
    """One-hot encode val against choices with an extra 'other' bin.

    Why one-hot instead of integer encoding: integer encoding (carbon=6,
    nitrogen=7) implies an ordinal relationship that doesn't exist chemically.
    One-hot gives each category an independent learnable weight vector.
    """
    enc = [0] * (len(choices) + 1)   # +1 for the catch-all "other" bin
    try:    enc[choices.index(val)] = 1
    except ValueError: enc[-1] = 1   # val not in choices → set "other" bit
    return enc

def atom_features(atom) -> list:
    """Compute 33-dimensional atom feature vector.

    Feature groups and why each matters:
      atomic number (11d): element identity — carbon vs nitrogen vs halogen
        fundamentally changes reactivity and binding mode.
      degree (7d): how many bonds — affects steric accessibility of the atom.
      formal charge (4d): charged atoms (e.g. quaternary N+) have very different
        electrostatic and binding properties than neutral atoms.
      H count (5d): implicit hydrogens affect H-bond donor capacity.
      hybridisation (5d): sp2 (aromatic/double bond) vs sp3 (tetrahedral) atoms
        have different geometry and reactivity.
      aromaticity (1d): aromatic atoms are planar, conjugated, and contribute
        to π-stacking interactions with protein residues.
    """
    return (
        onek(atom.GetAtomicNum(),     COMMON_ATOMS)   +  # 11-dim: element type
        onek(atom.GetTotalDegree(),   [0,1,2,3,4,5])  +  #  7-dim: bond count
        onek(atom.GetFormalCharge(),  [-1, 0, 1])     +  #  4-dim: charge state
        onek(atom.GetTotalNumHs(),    [0, 1, 2, 3])   +  #  5-dim: H neighbours
        onek(atom.GetHybridization(), HYB)            +  #  5-dim: orbital type
        [int(atom.GetIsAromatic())]                      #  1-dim: aromatic flag
    )  # total = 33 dimensions

ATOM_FDIM = 33

def bond_features(bond) -> list:
    """Compute 6-dimensional bond feature vector.

    Bond type as four binary flags (not one-hot) because only one type is ever
    active per bond. In-ring and conjugated flags capture delocalisation context
    that matters for electrophilicity and metabolic activation.
    """
    bt = bond.GetBondTypeAsDouble()   # 1.0=single, 2.0=double, 3.0=triple, 1.5=aromatic
    return [
        bt == 1.0,                    # single bond
        bt == 2.0,                    # double bond (electrophilic sites)
        bt == 3.0,                    # triple bond
        bt == 1.5,                    # aromatic bond
        bond.IsInRing(),              # ring membership — affects conformational rigidity
        bond.GetIsConjugated(),       # conjugation — affects electron delocalisation
    ]

BOND_FDIM = 6

def mol_to_graph(smi: str):
    """Convert a SMILES string into a directed graph dict for D-MPNN.

    Why directed edges: each bond A–B becomes TWO directed edges: A→B and B→A.
    We track which edge is the reverse of which via rev_idx. During message
    passing, when updating edge A→B, we SUBTRACT the message from B→A — this
    prevents an 'echo chamber' where B immediately reflects A's own message back,
    causing information to collapse to a trivial fixed point after a few steps.
    """
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None   # unparseable SMILES — caller skips this molecule

    atom_f = [atom_features(a) for a in mol.GetAtoms()]
    edge_src, edge_dst, bond_f, rev_idx = [], [], [], []

    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        n = len(edge_src)            # current number of directed edges added so far
        edge_src += [i, j]           # add forward edge i→j AND backward edge j→i
        edge_dst += [j, i]
        bond_f   += [bf, bf]         # both directions share the same bond features
        # rev_idx stores the index of the reverse partner:
        #   edge n   (i→j) has reverse = edge n+1 (j→i)
        #   edge n+1 (j→i) has reverse = edge n   (i→j)
        rev_idx  += [n + 1, n]

    # Edge case: single-atom molecule (e.g. noble gas, metal ion) has no bonds.
    # Add a placeholder self-loop so the model can still produce an embedding.
    if not edge_src:
        edge_src = [0]; edge_dst = [0]
        bond_f   = [[0] * BOND_FDIM]; rev_idx = [0]

    return {
        'atom_f':   torch.FloatTensor(atom_f),      # (N_atoms, 33)
        'edge_src': torch.LongTensor(edge_src),     # (N_edges,)  — source atom index
        'edge_dst': torch.LongTensor(edge_dst),     # (N_edges,)  — dest atom index
        'bond_f':   torch.FloatTensor(bond_f),      # (N_edges, 6)
        'rev':      torch.LongTensor(rev_idx),      # (N_edges,)  — reverse edge index
        'n_atoms':  mol.GetNumAtoms(),              # needed for scatter_add_ buffer size
    }

def build_graphs(smiles_list):
    """Build graph dicts for a list of SMILES, tracking which indices succeeded."""
    graphs, valid_idx = [], []
    for i, smi in enumerate(smiles_list):
        g = mol_to_graph(smi)
        if g is not None:
            graphs.append(g)
            valid_idx.append(i)   # track original index to align labels later
    return graphs, np.array(valid_idx)

print('Building molecular graph datasets...')
train_graphs, train_vidx = build_graphs(smiles_train)
valid_graphs, valid_vidx = build_graphs(smiles_valid)
test_graphs,  test_vidx  = build_graphs(smiles_test)
print(f'Graphs: train={len(train_graphs):,} | valid={len(valid_graphs):,} | test={len(test_graphs):,}')

In [ ]:
# =============================================================================
# CELL 7 — D-MPNN model + training (depth=5, hidden=300, focal loss, 30 epochs)
# =============================================================================

class DMPNN(nn.Module):
    """Directed Message Passing Neural Network (Yang et al. 2019 / chemprop).

    Three learnable weight matrices drive the algorithm:
      W_i: initialises each directed edge from atom features + bond features
      W_h: updates edge hidden states during message passing iterations
      W_o: combines final edge messages with atom features for atom-level readout
    """
    def __init__(self, atom_fdim: int, bond_fdim: int, n_tasks: int,
                 hidden: int = 300, depth: int = 5, dropout: float = 0.2):
        super().__init__()
        self.depth = depth   # number of message passing steps = receptive field in bonds

        # W_i: input projection — maps [atom_features || bond_features] → hidden.
        # No bias because BatchNorm (implicitly via the message passing update)
        # makes bias redundant in the first linear layer.
        self.W_i = nn.Linear(atom_fdim + bond_fdim, hidden, bias=False)

        # W_h: message update — maps aggregated neighbour messages → hidden.
        # Square matrix (hidden→hidden) so it can act as a recurrent-style update.
        self.W_h = nn.Linear(hidden, hidden, bias=False)

        # W_o: atom readout — combines original atom features with final edge messages.
        # Concatenating raw atom_f here is a skip connection: even after depth=5
        # message passes, the original atom identity is directly accessible.
        self.W_o = nn.Linear(atom_fdim + hidden, hidden)

        self.dropout = nn.Dropout(dropout)
        # Per-task FFN heads: 2-layer MLP (300→150→1) applied to the molecule embedding.
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden, hidden // 2), nn.ReLU(),
                          nn.Dropout(dropout), nn.Linear(hidden // 2, 1))
            for _ in range(n_tasks)
        ])

    def encode(self, g: dict) -> torch.Tensor:
        """Run D-MPNN message passing on a single molecule graph → 300-dim vector."""
        atom_f = g['atom_f'].to(DEVICE)    # (N_atoms, 33)
        bond_f = g['bond_f'].to(DEVICE)    # (N_edges,  6)
        src    = g['edge_src'].to(DEVICE)  # source atom index for each directed edge
        dst    = g['edge_dst'].to(DEVICE)  # destination atom index
        rev    = g['rev'].to(DEVICE)       # reverse edge index for each directed edge

        # Step 1: Initialise hidden state for every directed edge.
        # Concatenate the SOURCE atom's features with this edge's bond features,
        # then project to hidden-dim. This encodes "what atom is sending this message
        # and what type of bond is it travelling along?"
        h = torch.relu(self.W_i(torch.cat([atom_f[src], bond_f], dim=1)))   # (E, hidden)

        # Steps 2…T: Iterative directed message passing
        for _ in range(self.depth - 1):
            # agg[v] = sum of all hidden states from edges ARRIVING at atom v.
            # scatter_add_ is the GPU-efficient way to do indexed summation:
            # agg[dst[k]] += h[k] for all edges k. No Python loop needed.
            agg = torch.zeros(g['n_atoms'], h.shape[1], device=DEVICE)
            agg.scatter_add_(0, dst.unsqueeze(1).expand_as(h), h)

            # The DIRECTED trick: when computing the message for edge (u→v),
            # we sum all messages arriving at u EXCEPT the one from v (the reverse).
            # m = agg[src] gives ALL messages at src, then h[rev] removes the
            # reverse edge's contribution. This prevents a node from "hearing its
            # own echo" and forces information to propagate forward through the graph.
            m = agg[src] - h[rev]   # (E, hidden) — directed message, reverse excluded

            # Residual update: h_new = ReLU(h_old + W_h * m).
            # The skip connection (+h) is critical at depth=5: without it, gradients
            # must pass through 4 W_h matrices before reaching W_i, risking vanishing.
            # The residual provides a direct gradient highway back to early layers.
            h = torch.relu(h + self.W_h(m))

        # Step 3: Atom-level readout — aggregate final edge messages at each atom.
        atom_msg = torch.zeros(g['n_atoms'], h.shape[1], device=DEVICE)
        atom_msg.scatter_add_(0, dst.unsqueeze(1).expand_as(h), h)

        # Concatenate original atom features (skip connection) with aggregated messages,
        # then project to hidden-dim. The skip ensures element identity is never lost
        # even after 5 rounds of aggregation that might blur atom-specific information.
        atom_h = torch.relu(self.W_o(torch.cat([atom_f, atom_msg], dim=1)))  # (N, hidden)

        # Step 4: Sum pooling over all atoms → one fixed-size molecule embedding.
        # Sum (not mean) is used because molecule size carries information: larger
        # molecules naturally have more total signal, which is a relevant feature
        # for properties like molecular weight and lipophilicity.
        return atom_h.sum(0)   # (hidden,) — one vector per molecule

    def forward(self, graphs: list) -> torch.Tensor:
        # Encode each graph independently (variable-size, so no batching here),
        # then stack into a batch tensor for the parallel per-task head evaluation.
        mol_emb = torch.stack([self.encode(g) for g in graphs])   # (batch, hidden)
        # Apply dropout to the shared molecule embedding before the task heads —
        # this acts as a regulariser on the final representation, not per-head.
        return torch.cat([head(self.dropout(mol_emb)) for head in self.heads], dim=1)
        # output shape: (batch, 12) — one logit per endpoint


# -- Instantiate with deeper/wider hyperparameters ---------------------------
# depth=5 means each atom can "see" information from atoms up to 5 bonds away.
# This is needed to capture full pharmacophore patterns (e.g. the distance
# between an H-bond donor and acceptor across a rigid scaffold, often 4-6 bonds).
# hidden=300 (vs. default 256) provides more representational capacity for the
# shared message passing weights without a large compute increase.
dmpnn     = DMPNN(ATOM_FDIM, BOND_FDIM, len(tox21_tasks), hidden=300, depth=5).to(DEVICE)
opt_gnn   = torch.optim.Adam(dmpnn.parameters(), lr=3e-4, weight_decay=1e-5)
# Cosine schedule: lower lr (3e-4 vs 1e-3 for DeepTox) because D-MPNN has
# more complex geometry and benefits from a more conservative learning rate.
sched_gnn = torch.optim.lr_scheduler.CosineAnnealingLR(opt_gnn, T_max=30)

# Index label/weight matrices to only the graphs that parsed successfully.
# train_vidx maps graph index → original dataset row index.
y_gnn_tr = torch.FloatTensor(train_ds.y[train_vidx]).to(DEVICE)
w_gnn_tr = torch.FloatTensor(train_ds.w[train_vidx]).to(DEVICE)

EPOCHS_G = 30
BATCH_G  = 64   # smaller than DeepTox (256) because graph encoding is slower per sample
best_auc_gnn, best_state_gnn = 0.0, None

print(f'Training D-MPNN (depth=5, hidden=300, {EPOCHS_G} epochs, focal loss)...')
for epoch in range(EPOCHS_G):
    dmpnn.train(); ep_loss = 0.0; n_batches = 0

    # Manual batch loop over graphs (can't use DataLoader — graphs are variable-size dicts).
    # Shuffle via permutation each epoch to avoid ordering bias.
    perm = np.random.permutation(len(train_graphs))
    for start in range(0, len(train_graphs), BATCH_G):
        bi   = perm[start:start + BATCH_G]
        bg   = [train_graphs[j] for j in bi]   # list of graph dicts for this batch
        yb   = y_gnn_tr[bi]
        wb   = w_gnn_tr[bi]
        opt_gnn.zero_grad()
        # masked_focal defined in DeepTox cell — reused here for the same reason:
        # class imbalance and missing labels require the same treatment.
        loss = masked_focal(dmpnn(bg), yb, wb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(dmpnn.parameters(), 1.0)
        opt_gnn.step()
        ep_loss += loss.item(); n_batches += 1
    sched_gnn.step()

    dmpnn.eval()
    with torch.no_grad():
        vp_gnn = torch.sigmoid(dmpnn(valid_graphs)).cpu().numpy()
    y_va = valid_ds.y[valid_vidx]; w_va = valid_ds.w[valid_vidx]
    aucs_g = []
    for i in range(len(tox21_tasks)):
        m = w_va[:, i] > 0; lbl = y_va[:, i][m]
        if m.sum() >= 10 and len(np.unique(lbl)) > 1:
            try: aucs_g.append(roc_auc_score(lbl, vp_gnn[:, i][m]))
            except: pass
    mean_va_g = float(np.mean(aucs_g)) if aucs_g else 0.0
    if mean_va_g > best_auc_gnn:
        best_auc_gnn = mean_va_g
        best_state_gnn = {k: v.clone() for k, v in dmpnn.state_dict().items()}
    if epoch % 5 == 0 or epoch == EPOCHS_G - 1:
        print(f'  Epoch {epoch:2d} | Loss={ep_loss/n_batches:.4f} | '
              f'Val AUC={mean_va_g:.4f} | LR={opt_gnn.param_groups[0]["lr"]:.2e}')

# Restore best checkpoint and evaluate on test
dmpnn.load_state_dict(best_state_gnn)
dmpnn.eval()
with torch.no_grad():
    tp_gnn = torch.sigmoid(dmpnn(test_graphs)).cpu().numpy()
y_te = test_ds.y[test_vidx]; w_te = test_ds.w[test_vidx]
dmpnn_aucs = {}
for i, task in enumerate(tox21_tasks):
    m = w_te[:, i] > 0; lbl = y_te[:, i][m]
    if m.sum() >= 5 and len(np.unique(lbl)) > 1:
        try: dmpnn_aucs[task] = roc_auc_score(lbl, tp_gnn[:, i][m])
        except: pass
print(f'D-MPNN mean test AUC: {np.mean(list(dmpnn_aucs.values())):.4f}')

## Head-to-Head Comparison

In [ ]:
# =============================================================================
# CELL 8 — Head-to-head comparison: per-task AUC table + grouped bar chart
# =============================================================================

methods = {'Tox21NN': tox21nn_aucs, 'DeepTox': deeptox_aucs, 'D-MPNN': dmpnn_aucs}

# Build a per-task comparison DataFrame.
# float('nan') is inserted for tasks where AUC couldn't be computed (e.g. a
# test fold that happened to contain only negatives for a rare endpoint).
rows = []
for task in tox21_tasks:
    row = {'Endpoint': task}
    for m, d in methods.items():
        row[m] = round(d.get(task, float('nan')), 4)
    rows.append(row)
comp_df = pd.DataFrame(rows)

# idxmax(axis=1) returns the column name with the highest value per row —
# gives a quick "which method wins on this endpoint?" column.
comp_df['Best'] = comp_df[['Tox21NN', 'DeepTox', 'D-MPNN']].idxmax(axis=1)
print(comp_df.to_string(index=False))

# np.nanmean ignores NaN entries — so endpoints where a method couldn't compute
# AUC don't drag the mean down to 0 for that method.
means = {m: np.nanmean(list(d.values())) for m, d in methods.items()}
print('\nMean test AUC (scaffold split):')
for m, v in sorted(means.items(), key=lambda x: -x[1]):
    print(f'  {m:12s}: {v:.4f}  {"|" * int(v * 40)}')   # ASCII bar proportional to AUC

# -- Grouped bar chart --------------------------------------------------------
# Each group of 3 bars = one Tox21 endpoint.
# Bars within a group are the three methods, coloured consistently.
# This makes it easy to see which method wins endpoint-by-endpoint.
x = np.arange(len(tox21_tasks))
w = 0.27   # bar width — 3 bars × 0.27 ≈ 0.81 < 1.0 (no overlap within group)
colors = {'Tox21NN': '#95a5a6', 'DeepTox': '#2980b9', 'D-MPNN': '#27ae60'}

fig, ax = plt.subplots(figsize=(14, 5))
for k, (method, d) in enumerate(methods.items()):
    vals = [d.get(t, float('nan')) for t in tox21_tasks]
    ax.bar(x + k * w, vals, w,
           label=f'{method} (mean={means[method]:.3f})',
           color=colors[method], alpha=0.85, edgecolor='white', linewidth=0.3)

ax.set_xticks(x + w)
ax.set_xticklabels(tox21_tasks, rotation=45, ha='right', fontsize=8)
# AUC=0.80 guideline: EPA and FDA use this as a minimum threshold for
# accepting computational models in regulatory submissions (ToxCast, ICH S2).
ax.axhline(0.8, color='k', linestyle='--', lw=0.7, label='AUC = 0.80 (regulatory guideline)')
ax.set_ylabel('Test ROC-AUC')
ax.set_ylim([0.5, 1.0])
ax.set_title('Tox21 Benchmark — Tox21NN vs DeepTox vs D-MPNN (scaffold split)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('tox21_comparison.png', dpi=150)
plt.show()

## Key Takeaways

### Method insights
- **Tox21NN** (per-task MLP): fast and simple, but each endpoint is solved in isolation — rare endpoints with few positives receive no benefit from related tasks.
- **DeepTox** (multi-task MLP): shared representation improves rare endpoints (NR-AR, SR-ATAD5) by propagating cross-task signal; 2014 state-of-the-art.
- **D-MPNN** (graph-based): learns atom/bond interaction patterns directly from molecular topology — no hard radius-2 cutoff, no structural vocabulary limit. Typically matches or exceeds fingerprint methods on scaffold splits.

### Data & evaluation
- **Scaffold split** is mandatory: random split inflates AUC 5–15% by placing structural analogues in both train and test.
- **Missing labels (`w=0`) must be masked** from loss and AUC: imputing zeros doubles apparent negatives and destroys calibration for rare endpoints.
- **Mean AUC** should be computed over tasks with sufficient data only (`n_measured ≥ 5`, both classes present).

### Published benchmarks
| Model | Mean AUC (scaffold) | Year |
|-------|---------------------|------|
| Tox21NN | ~0.790 | 2014 |
| DeepTox | ~0.846 | 2016 |
| chemprop D-MPNN | ~0.855 | 2019 |
| AttentiveFP | ~0.862 | 2020 |
| Uni-Mol | ~0.875 | 2023 |

### Industry context
Regulatory submissions (EPA CompTox, FDA ToxCast, ICH S2) increasingly accept computational Tox21-class models as tier-1 screening when AUC ≥ 0.80 with calibrated uncertainty estimates.